In [ ]:
import struct
import socket
import zlib
import time
import threading
import os
from enum import Enum
from dataclasses import dataclass
from typing import Optional, Tuple, Dict

In [ ]:
# Constantes do protocolo SRTP

HEADER_SIZE = 9  # bytes
MAX_PAYLOAD = 255  # bytes por pacote
MAX_SEQ = 2 ** 14  # 14 bits -> 0 a 16383
TIMEOUT = 0.1  # 100ms
MIN_WINDOW = 1
MAX_WINDOW = 255
DEFAULT_WINDOW = 16

class ProtocolMode(Enum):
    SAW = "saw"      # Stop-and-Wait
    GBN = "gbn"      # Go-Back-N
    SR = "sr"        # Selective Repeat

class PacketFlags:
    SYN = 0x80       # bit 7
    FIN = 0x40       # bit 6
    ACK_FLAG = 0x02  # bit 1 (para marcar que ACK é válido)
    NACK = 0x01      # bit 0

In [ ]:
@dataclass
class SrtpHeader:
    """Estrutura do cabeçalho SRTP (9 bytes)"""
    seq: int = 0          # 14 bits: número de sequência
    syn: bool = False     # 1 bit: flag SYN
    fin: bool = False     # 1 bit: flag FIN
    ack: int = 0          # 14 bits: número de acknowledgement
    ack_flag: bool = False    # 1 bit: ACK válido?
    nack: bool = False    # 1 bit: negative acknowledgement
    length: int = 0       # 8 bits: tamanho do payload ou janela no handshake
    crc32: int = 0        # 32 bits: checksum CRC32

def build_header(seq: int, ack: int, length: int, syn: bool = False, fin: bool = False, 
                 ack_flag: bool = False, nack: bool = False) -> bytes:
    """
    Constrói o cabeçalho SRTP (9 bytes).
    
    Layout:
    Byte 0-1: |S|F|SEQ (14 bits)|
    Byte 2-3: |A|N|ACK (14 bits)|
    Byte 4:   |Length (8 bits)|
    Byte 5-8: |CRC32 (32 bits)|
    """
    # Byte 0-1: S (1 bit) + F (1 bit) + SEQ (14 bits)
    flags1 = (int(syn) << 7) | (int(fin) << 6)
    word1 = (flags1 << 14) | (seq & 0x3FFF)
    
    # Byte 2-3: A (1 bit) + N (1 bit) + ACK (14 bits)
    flags2 = (int(ack_flag) << 7) | (int(nack) << 6)
    word2 = (flags2 << 14) | (ack & 0x3FFF)
    
    # Bytes 0-3: cabeçalho sem CRC (com CRC zerado)
    header_no_crc = struct.pack('!HH', word1, word2) + struct.pack('!B', length & 0xFF)
    
    # Retorna header com CRC32 zerado para posterior cálculo
    return header_no_crc + b'\x00\x00\x00\x00'

def parse_header(data: bytes) -> Optional[SrtpHeader]:
    """Desserializa o cabeçalho SRTP de 9 bytes."""
    if len(data) < HEADER_SIZE:
        return None
    
    word1, word2, length = struct.unpack('!HHB', data[0:5])
    crc32_bytes = struct.unpack('!I', data[5:9])
    
    # Extrai flags e SEQ do primeiro word
    syn = bool(word1 & 0x8000)
    fin = bool(word1 & 0x4000)
    seq = word1 & 0x3FFF
    
    # Extrai flags e ACK do segundo word
    ack_flag = bool(word2 & 0x8000)
    nack = bool(word2 & 0x4000)
    ack = word2 & 0x3FFF
    
    crc32_val = crc32_bytes[0]
    
    return SrtpHeader(seq=seq, syn=syn, fin=fin, ack=ack, 
                      ack_flag=ack_flag, nack=nack, length=length, crc32=crc32_val)

print("Funções de parser e montagem do cabeçalho SRTP definidas.")

In [ ]:
def calculate_crc32(header_no_crc: bytes, payload: bytes = b'') -> int:
    """
    Calcula CRC32 sobre o cabeçalho (com CRC32 zerado) + payload.
    
    O campo CRC32 no cabeçalho deve estar zerado antes do cálculo.
    """
    data = header_no_crc + payload
    crc = zlib.crc32(data) & 0xFFFFFFFF
    return crc

def add_crc_to_packet(header_no_crc: bytes, payload: bytes = b'') -> bytes:
    """Adiciona o CRC32 calculado ao cabeçalho e retorna o pacote completo."""
    crc = calculate_crc32(header_no_crc, payload)
    packet = header_no_crc[:5] + struct.pack('!I', crc) + payload
    return packet

def verify_crc32(packet: bytes) -> Tuple[bool, Optional[bytes]]:
    """
    Verifica se o CRC32 do pacote é válido.
    
    Retorna: (válido, payload)
    Se inválido, retorna (False, None).
    """
    if len(packet) < HEADER_SIZE:
        return False, None
    
    header = packet[:5]
    crc32_received = struct.unpack('!I', packet[5:9])[0]
    payload = packet[HEADER_SIZE:] if len(packet) > HEADER_SIZE else b''
    
    # Recalcula CRC com o campo zerado
    crc32_calculated = calculate_crc32(header, payload)
    
    if crc32_calculated == crc32_received:
        return True, payload
    else:
        return False, None

# Teste de CRC32
test_header = build_header(seq=5, ack=0, length=10)
test_payload = b'Hello World!'[:10]
test_packet = add_crc_to_packet(test_header, test_payload)

valid, extracted_payload = verify_crc32(test_packet)
print(f"Teste CRC32: Pacote válido = {valid}, Payload extraído = {extracted_payload}")

In [ ]:
class SrtpConnection:
    """Gerencia uma conexão SRTP com handshake e encerramento."""
    
    def __init__(self, mode: ProtocolMode = ProtocolMode.SAW, window_size: int = DEFAULT_WINDOW):
        self.mode = mode
        self.window_size = window_size
        self.remote_window = 0
        self.connected = False
        self.seq_out = 0
        self.seq_in = 0
    
    def create_syn_packet(self, window_size: int = DEFAULT_WINDOW) -> bytes:
        """Cria pacote SYN (iniciador para receiver)."""
        header = build_header(seq=0, ack=0, length=window_size, 
                             syn=True, fin=False, ack_flag=False, nack=False)
        packet = add_crc_to_packet(header)
        return packet
    
    def create_syn_ack_packet(self, remote_window: int, receiver_window: int) -> bytes:
        """Cria pacote SYN+ACK (receiver para iniciador)."""
        # Armazena janela remota
        self.remote_window = remote_window
        
        header = build_header(seq=0, ack=0, length=receiver_window,
                             syn=True, fin=True, ack_flag=True, nack=False)
        packet = add_crc_to_packet(header)
        return packet
    
    def create_ack_packet(self) -> bytes:
        """Cria pacote ACK de confirmação (iniciador para receiver)."""
        header = build_header(seq=0, ack=0, length=0,
                             syn=False, fin=False, ack_flag=True, nack=False)
        packet = add_crc_to_packet(header)
        return packet
    
    def process_handshake(self, packet: bytes, is_initiator: bool) -> Optional[int]:
        """
        Processa pacote de handshake.
        Retorna a janela negociada (menor das duas propostas) ou None se falhar.
        """
        valid, _ = verify_crc32(packet)
        if not valid:
            return None
        
        header = parse_header(packet)
        if header is None:
            return None
        
        if is_initiator:
            # Esperamos SYN+ACK
            if header.syn and header.fin:
                self.remote_window = header.length
                self.window_size = min(self.window_size, self.remote_window)
                self.connected = True
                return self.window_size
        else:
            # Esperamos SYN
            if header.syn and not header.fin:
                self.remote_window = header.length
                self.window_size = min(self.window_size, self.remote_window)
                return self.window_size
        
        return None

print("Classe SrtpConnection com handshake definida.")

In [ ]:
class StopAndWaitSender(SrtpConnection):
    """Implementa transferência de arquivo com Stop-and-Wait."""
    
    def __init__(self):
        super().__init__(mode=ProtocolMode.SAW, window_size=1)
        self.retransmissions = 0
        self.packets_sent = 0
    
    def send_file(self, sock: socket.socket, remote_addr: Tuple[str, int], 
                  file_path: str) -> bool:
        """
        Envia arquivo com stop-and-wait.
        
        Retorna True se sucesso, False se falhar.
        """
        if not os.path.exists(file_path):
            print(f"Erro: arquivo {file_path} não encontrado.")
            return False
        
        try:
            with open(file_path, 'rb') as f:
                file_data = f.read()
            
            file_size = len(file_data)
            bytes_sent = 0
            seq = 0
            
            while bytes_sent < file_size:
                # Prepara o próximo pacote
                chunk_size = min(MAX_PAYLOAD, file_size - bytes_sent)
                payload = file_data[bytes_sent:bytes_sent + chunk_size]
                
                # Cria header do pacote de dados
                header = build_header(seq=seq, ack=0, length=len(payload),
                                     syn=False, fin=False, ack_flag=False, nack=False)
                packet = add_crc_to_packet(header, payload)
                
                # Envia e aguarda ACK com timeout
                last_ack_seq = -1
                while True:
                    sock.sendto(packet, remote_addr)
                    self.packets_sent += 1
                    
                    sock.settimeout(TIMEOUT)
                    try:
                        ack_packet, _ = sock.recvfrom(HEADER_SIZE)
                        valid, _ = verify_crc32(ack_packet)
                        
                        if valid:
                            ack_header = parse_header(ack_packet)
                            if ack_header and ack_header.ack_flag and ack_header.ack == seq:
                                last_ack_seq = seq
                                break  # ACK recebido, próximo pacote
                            
                    except socket.timeout:
                        # Timeout, retransmite
                        self.retransmissions += 1
                        continue
                
                seq = (seq + 1) % MAX_SEQ
                bytes_sent += chunk_size
            
            print(f"Arquivo enviado: {bytes_sent} bytes, {self.packets_sent} pacotes, "
                  f"{self.retransmissions} retransmissões")
            return True
            
        except Exception as e:
            print(f"Erro ao enviar arquivo: {e}")
            return False

class StopAndWaitReceiver(SrtpConnection):
    """Implementa recebimento de arquivo com Stop-and-Wait."""
    
    def __init__(self):
        super().__init__(mode=ProtocolMode.SAW, window_size=1)
        self.packets_received = 0
    
    def receive_file(self, sock: socket.socket, output_path: str) -> bool:
        """
        Recebe arquivo com stop-and-wait.
        
        Retorna True se sucesso, False se falhar.
        """
        try:
            file_data = b''
            expected_seq = 0
            
            while True:
                sock.settimeout(5.0)  # timeout maior para permitir envio de dados
                
                try:
                    packet, remote_addr = sock.recvfrom(HEADER_SIZE + MAX_PAYLOAD)
                    
                    # Verifica CRC
                    valid, payload = verify_crc32(packet)
                    if not valid:
                        print(f"Pacote com CRC inválido descartado.")
                        continue  # Descarta silenciosamente
                    
                    # Parse do header
                    header = parse_header(packet)
                    if header is None:
                        continue
                    
                    # Verifica sequência
                    if header.seq != expected_seq:
                        print(f"Pacote fora de ordem descartado (esperado {expected_seq}, recebido {header.seq})")
                        continue  # Descarta silenciosamente
                    
                    # Acumula dados
                    if payload:
                        file_data += payload
                    
                    self.packets_received += 1
                    
                    # Envia ACK
                    ack_header = build_header(seq=0, ack=expected_seq, length=0,
                                            syn=False, fin=False, ack_flag=True, nack=False)
                    ack_packet = add_crc_to_packet(ack_header)
                    sock.sendto(ack_packet, remote_addr)
                    
                    # Verifica se este é o último pacote
                    if header.length < MAX_PAYLOAD:
                        # Último pacote recebido
                        break
                    
                    expected_seq = (expected_seq + 1) % MAX_SEQ
                    
                except socket.timeout:
                    if file_data:
                        break  # Timeout após receber dados = fim
                    continue
            
            # Salva o arquivo
            with open(output_path, 'wb') as f:
                f.write(file_data)
            
            print(f"Arquivo recebido: {len(file_data)} bytes, {self.packets_received} pacotes")
            return True
            
        except Exception as e:
            print(f"Erro ao receber arquivo: {e}")
            return False

print("Implementação de Stop-and-Wait definida.")

In [ ]:
class GoBackNSender(SrtpConnection):
    """Implementa transferência com Go-Back-N (GBN)."""
    
    def __init__(self, window_size: int = DEFAULT_WINDOW):
        super().__init__(mode=ProtocolMode.GBN, window_size=min(window_size, MAX_WINDOW))
        self.retransmissions = 0
        self.packets_sent = 0
        self.packets_in_flight: Dict[int, bytes] = {}  # Pacotes na janela
        self.base_seq = 0  # Primeiro pacote não confirmado
    
    def send_file(self, sock: socket.socket, remote_addr: Tuple[str, int], 
                  file_path: str) -> bool:
        """Envia arquivo com GBN."""
        if not os.path.exists(file_path):
            print(f"Erro: arquivo {file_path} não encontrado.")
            return False
        
        try:
            with open(file_path, 'rb') as f:
                file_data = f.read()
            
            file_size = len(file_data)
            bytes_sent = 0
            seq = 0
            packets_to_send = []
            
            # Prepara todos os pacotes
            while bytes_sent < file_size:
                chunk_size = min(MAX_PAYLOAD, file_size - bytes_sent)
                payload = file_data[bytes_sent:bytes_sent + chunk_size]
                header = build_header(seq=seq, ack=0, length=len(payload),
                                     syn=False, fin=False, ack_flag=False, nack=False)
                packet = add_crc_to_packet(header, payload)
                packets_to_send.append((seq, packet))
                
                seq = (seq + 1) % MAX_SEQ
                bytes_sent += chunk_size
            
            # Envia pacotes em janela
            next_to_send = 0
            self.base_seq = 0
            
            while self.base_seq < len(packets_to_send):
                # Preenche a janela
                while (next_to_send - self.base_seq) < self.window_size and next_to_send < len(packets_to_send):
                    seq_num, packet = packets_to_send[next_to_send]
                    sock.sendto(packet, remote_addr)
                    self.packets_in_flight[seq_num] = packet
                    self.packets_sent += 1
                    next_to_send += 1
                
                # Aguarda ACK com timeout
                sock.settimeout(TIMEOUT)
                try:
                    ack_packet, _ = sock.recvfrom(HEADER_SIZE)
                    valid, _ = verify_crc32(ack_packet)
                    
                    if valid:
                        ack_header = parse_header(ack_packet)
                        if ack_header and ack_header.ack_flag:
                            # ACK cumulativo: confirma todos os pacotes até ack_header.ack
                            if ack_header.nack:
                                # NACK recebido: retransmite a partir do ack_header.ack
                                self.base_seq = ack_header.ack
                                self.retransmissions += (next_to_send - self.base_seq)
                            else:
                                # ACK normal
                                self.base_seq = (ack_header.ack + 1) % MAX_SEQ
                                # Remove pacotes confirmados
                                for seq_to_remove in list(self.packets_in_flight.keys()):
                                    if (seq_to_remove - self.base_seq) % MAX_SEQ >= self.window_size:
                                        del self.packets_in_flight[seq_to_remove]
                
                except socket.timeout:
                    # Retransmite toda a janela
                    for i in range(self.base_seq, min(self.base_seq + self.window_size, len(packets_to_send))):
                        _, packet = packets_to_send[i % len(packets_to_send)]
                        sock.sendto(packet, remote_addr)
                        self.retransmissions += 1
            
            print(f"Arquivo enviado (GBN): {bytes_sent} bytes, {self.packets_sent} pacotes, "
                  f"{self.retransmissions} retransmissões, janela={self.window_size}")
            return True
            
        except Exception as e:
            print(f"Erro ao enviar arquivo (GBN): {e}")
            return False

class GoBackNReceiver(SrtpConnection):
    """Implementa recebimento com Go-Back-N."""
    
    def __init__(self, window_size: int = DEFAULT_WINDOW):
        super().__init__(mode=ProtocolMode.GBN, window_size=min(window_size, MAX_WINDOW))
        self.packets_received = 0
        self.expected_seq = 0
    
    def receive_file(self, sock: socket.socket, output_path: str) -> bool:
        """Recebe arquivo com GBN."""
        try:
            file_data = b''
            self.expected_seq = 0
            
            while True:
                sock.settimeout(5.0)
                
                try:
                    packet, remote_addr = sock.recvfrom(HEADER_SIZE + MAX_PAYLOAD)
                    
                    valid, payload = verify_crc32(packet)
                    if not valid:
                        continue  # Descarta silenciosamente
                    
                    header = parse_header(packet)
                    if header is None:
                        continue
                    
                    # GBN: descarta pacotes fora de ordem
                    if header.seq != self.expected_seq:
                        # Envia NACK com o pacote esperado
                        nack_header = build_header(seq=0, ack=self.expected_seq, length=0,
                                                 syn=False, fin=False, ack_flag=True, nack=True)
                        nack_packet = add_crc_to_packet(nack_header)
                        sock.sendto(nack_packet, remote_addr)
                        continue  # Descarta pacote fora de ordem
                    
                    # Pacote esperado
                    if payload:
                        file_data += payload
                    
                    self.packets_received += 1
                    
                    # Envia ACK cumulativo
                    ack_header = build_header(seq=0, ack=header.seq, length=0,
                                            syn=False, fin=False, ack_flag=True, nack=False)
                    ack_packet = add_crc_to_packet(ack_header)
                    sock.sendto(ack_packet, remote_addr)
                    
                    if header.length < MAX_PAYLOAD:
                        break
                    
                    self.expected_seq = (self.expected_seq + 1) % MAX_SEQ
                    
                except socket.timeout:
                    if file_data:
                        break
                    continue
            
            with open(output_path, 'wb') as f:
                f.write(file_data)
            
            print(f"Arquivo recebido (GBN): {len(file_data)} bytes, {self.packets_received} pacotes")
            return True
            
        except Exception as e:
            print(f"Erro ao receber arquivo (GBN): {e}")
            return False

print("Implementação de Go-Back-N definida.")

In [ ]:
class SelectiveRepeatSender(SrtpConnection):
    """Implementa transferência com Selective Repeat (SR)."""
    
    def __init__(self, window_size: int = DEFAULT_WINDOW):
        super().__init__(mode=ProtocolMode.SR, window_size=min(window_size, MAX_WINDOW))
        self.retransmissions = 0
        self.packets_sent = 0
        self.packets_in_flight: Dict[int, bytes] = {}
        self.base_seq = 0
        self.ack_status: Dict[int, bool] = {}  # Rastreia ACKs individuais
    
    def send_file(self, sock: socket.socket, remote_addr: Tuple[str, int], 
                  file_path: str) -> bool:
        """Envia arquivo com SR."""
        if not os.path.exists(file_path):
            print(f"Erro: arquivo {file_path} não encontrado.")
            return False
        
        try:
            with open(file_path, 'rb') as f:
                file_data = f.read()
            
            file_size = len(file_data)
            bytes_sent = 0
            seq = 0
            packets_to_send = []
            
            # Prepara todos os pacotes
            while bytes_sent < file_size:
                chunk_size = min(MAX_PAYLOAD, file_size - bytes_sent)
                payload = file_data[bytes_sent:bytes_sent + chunk_size]
                header = build_header(seq=seq, ack=0, length=len(payload),
                                     syn=False, fin=False, ack_flag=False, nack=False)
                packet = add_crc_to_packet(header, payload)
                packets_to_send.append((seq, packet))
                
                seq = (seq + 1) % MAX_SEQ
                bytes_sent += chunk_size
            
            # Envia pacotes em janela
            next_to_send = 0
            self.base_seq = 0
            
            while self.base_seq < len(packets_to_send):
                # Preenche a janela
                while (next_to_send - self.base_seq) < self.window_size and next_to_send < len(packets_to_send):
                    seq_num, packet = packets_to_send[next_to_send]
                    sock.sendto(packet, remote_addr)
                    self.packets_in_flight[seq_num] = packet
                    self.ack_status[seq_num] = False
                    self.packets_sent += 1
                    next_to_send += 1
                
                # Aguarda ACK com timeout
                sock.settimeout(TIMEOUT)
                try:
                    ack_packet, _ = sock.recvfrom(HEADER_SIZE)
                    valid, _ = verify_crc32(ack_packet)
                    
                    if valid:
                        ack_header = parse_header(ack_packet)
                        if ack_header and ack_header.ack_flag:
                            if ack_header.nack:
                                # NACK recebido: retransmite apenas o pacote específico
                                self.retransmissions += 1
                                if ack_header.ack in self.packets_in_flight:
                                    _, packet = packets_to_send[ack_header.ack % len(packets_to_send)]
                                    sock.sendto(packet, remote_addr)
                            else:
                                # ACK individual
                                self.ack_status[ack_header.ack] = True
                                
                                # Move base_seq se pacotes consecutivos foram ACK'ed
                                while self.base_seq < len(packets_to_send) and self.ack_status.get(self.base_seq, False):
                                    if self.base_seq in self.packets_in_flight:
                                        del self.packets_in_flight[self.base_seq]
                                    if self.base_seq in self.ack_status:
                                        del self.ack_status[self.base_seq]
                                    self.base_seq += 1
                
                except socket.timeout:
                    # Retransmite apenas pacotes não-ACK'ed na janela
                    for seq_to_retrans in list(self.packets_in_flight.keys()):
                        if not self.ack_status.get(seq_to_retrans, False):
                            idx = seq_to_retrans % len(packets_to_send)
                            _, packet = packets_to_send[idx]
                            sock.sendto(packet, remote_addr)
                            self.retransmissions += 1
            
            print(f"Arquivo enviado (SR): {bytes_sent} bytes, {self.packets_sent} pacotes, "
                  f"{self.retransmissions} retransmissões, janela={self.window_size}")
            return True
            
        except Exception as e:
            print(f"Erro ao enviar arquivo (SR): {e}")
            return False

class SelectiveRepeatReceiver(SrtpConnection):
    """Implementa recebimento com Selective Repeat."""
    
    def __init__(self, window_size: int = DEFAULT_WINDOW):
        super().__init__(mode=ProtocolMode.SR, window_size=min(window_size, MAX_WINDOW))
        self.packets_received = 0
        self.expected_seq = 0
        self.buffered_packets: Dict[int, bytes] = {}  # Buffer de pacotes fora de ordem
    
    def receive_file(self, sock: socket.socket, output_path: str) -> bool:
        """Recebe arquivo com SR."""
        try:
            file_data = b''
            self.expected_seq = 0
            self.buffered_packets = {}
            
            while True:
                sock.settimeout(5.0)
                
                try:
                    packet, remote_addr = sock.recvfrom(HEADER_SIZE + MAX_PAYLOAD)
                    
                    valid, payload = verify_crc32(packet)
                    if not valid:
                        continue  # Descarta silenciosamente
                    
                    header = parse_header(packet)
                    if header is None:
                        continue
                    
                    # SR: aceita e bufferiza pacotes fora de ordem
                    if header.seq == self.expected_seq:
                        # Pacote esperado
                        if payload:
                            file_data += payload
                        self.packets_received += 1
                        
                        # Envia ACK individual
                        ack_header = build_header(seq=0, ack=header.seq, length=0,
                                                syn=False, fin=False, ack_flag=True, nack=False)
                        ack_packet = add_crc_to_packet(ack_header)
                        sock.sendto(ack_packet, remote_addr)
                        
                        if header.length < MAX_PAYLOAD:
                            # Processa buffer se há pacotes subsequentes
                            self.expected_seq = (self.expected_seq + 1) % MAX_SEQ
                            while self.expected_seq in self.buffered_packets:
                                buffered_payload = self.buffered_packets.pop(self.expected_seq)
                                file_data += buffered_payload
                                self.expected_seq = (self.expected_seq + 1) % MAX_SEQ
                            break
                        
                        self.expected_seq = (self.expected_seq + 1) % MAX_SEQ
                        # Processa buffer se há pacotes subsequentes
                        while self.expected_seq in self.buffered_packets:
                            buffered_payload = self.buffered_packets.pop(self.expected_seq)
                            file_data += buffered_payload
                            self.expected_seq = (self.expected_seq + 1) % MAX_SEQ
                    
                    elif (header.seq - self.expected_seq) % MAX_SEQ < self.window_size:
                        # Pacote dentro da janela mas fora de ordem
                        self.buffered_packets[header.seq] = payload if payload else b''
                        
                        # Envia ACK individual
                        ack_header = build_header(seq=0, ack=header.seq, length=0,
                                                syn=False, fin=False, ack_flag=True, nack=False)
                        ack_packet = add_crc_to_packet(ack_header)
                        sock.sendto(ack_packet, remote_addr)
                    
                    else:
                        # Pacote fora da janela: envia NACK do esperado
                        nack_header = build_header(seq=0, ack=self.expected_seq, length=0,
                                                 syn=False, fin=False, ack_flag=True, nack=True)
                        nack_packet = add_crc_to_packet(nack_header)
                        sock.sendto(nack_packet, remote_addr)
                    
                except socket.timeout:
                    if file_data:
                        break
                    continue
            
            with open(output_path, 'wb') as f:
                f.write(file_data)
            
            print(f"Arquivo recebido (SR): {len(file_data)} bytes, {self.packets_received} pacotes")
            return True
            
        except Exception as e:
            print(f"Erro ao receber arquivo (SR): {e}")
            return False

print("Implementação de Selective Repeat definida.")

In [ ]:
def create_fin_packet() -> bytes:
    """Cria pacote FIN (sender para receiver)."""
    header = build_header(seq=0, ack=0, length=0,
                         syn=False, fin=True, ack_flag=False, nack=False)
    packet = add_crc_to_packet(header)
    return packet

def create_fin_ack_packet() -> bytes:
    """Cria pacote FIN+ACK (receiver para sender)."""
    header = build_header(seq=0, ack=0, length=0,
                         syn=False, fin=True, ack_flag=True, nack=False)
    packet = add_crc_to_packet(header)
    return packet

def send_fin(sock: socket.socket, remote_addr: Tuple[str, int], max_retries: int = 3) -> bool:
    """Envia FIN e aguarda FIN+ACK."""
    fin_packet = create_fin_packet()
    
    for attempt in range(max_retries):
        sock.sendto(fin_packet, remote_addr)
        sock.settimeout(TIMEOUT)
        
        try:
            fin_ack_packet, _ = sock.recvfrom(HEADER_SIZE)
            valid, _ = verify_crc32(fin_ack_packet)
            
            if valid:
                header = parse_header(fin_ack_packet)
                if header and header.fin and header.ack_flag:
                    return True
        except socket.timeout:
            continue
    
    return False

def receive_fin(sock: socket.socket, remote_addr: Tuple[str, int]) -> bool:
    """Recebe FIN e envia FIN+ACK."""
    try:
        sock.settimeout(5.0)
        fin_packet, _ = sock.recvfrom(HEADER_SIZE)
        
        valid, _ = verify_crc32(fin_packet)
        if valid:
            header = parse_header(fin_packet)
            if header and header.fin and not header.ack_flag:
                fin_ack_packet = create_fin_ack_packet()
                sock.sendto(fin_ack_packet, remote_addr)
                return True
    except socket.timeout:
        pass
    
    return False

print("Funções de encerramento de conexão definidas.")

In [ ]:
import argparse
import hashlib

def compute_file_hash(file_path: str) -> str:
    """Calcula hash SHA-256 do arquivo para verificação de integridade."""
    sha256_hash = hashlib.sha256()
    with open(file_path, 'rb') as f:
        for byte_block in iter(lambda: f.read(4096), b''):
            sha256_hash.update(byte_block)
    return sha256_hash.hexdigest()

def main():
    """Interface de linha de comando para SRTP."""
    parser = argparse.ArgumentParser(description='SRTP - Simple Reliable Transport Protocol')
    
    # Argumentos comuns
    parser.add_argument('--port', type=int, required=True, help='Porta base P')
    parser.add_argument('--mode', choices=['saw', 'gbn', 'sr'], default='saw',
                       help='Modo de protocolo (default: saw)')
    parser.add_argument('--window', type=int, default=DEFAULT_WINDOW,
                       help='Tamanho da janela para GBN/SR (default: 16)')
    
    # Modo listen (receiver)
    parser.add_argument('--listen', action='store_true', help='Ativar modo listen (receiver)')
    
    # Modo connect (sender)
    parser.add_argument('--host', type=str, help='Endereço IP do receiver')
    parser.add_argument('--file', type=str, help='Arquivo a transferir')
    
    args = parser.parse_args()
    
    # Valida argumentos
    if not args.listen and (not args.host or not args.file):
        parser.error('Modo sender requer --host e --file')
    
    if args.listen and (args.host or args.file):
        parser.error('Modo listen não aceita --host ou --file')
    
    # Seleciona classe apropriada
    protocol_mode = ProtocolMode(args.mode)
    
    if protocol_mode == ProtocolMode.SAW:
        if args.listen:
            sender_class, receiver_class = None, StopAndWaitReceiver
        else:
            sender_class, receiver_class = StopAndWaitSender, None
    elif protocol_mode == ProtocolMode.GBN:
        if args.listen:
            sender_class, receiver_class = None, GoBackNReceiver
            receiver = receiver_class(args.window)
        else:
            sender_class = GoBackNSender
            sender = sender_class(args.window)
            receiver_class = None
    else:  # SR
        if args.listen:
            sender_class, receiver_class = None, SelectiveRepeatReceiver
            receiver = receiver_class(args.window)
        else:
            sender_class = SelectiveRepeatSender
            sender = sender_class(args.window)
            receiver_class = None
    
    # Executa sender ou receiver
    try:
        if args.listen:
            # Modo receiver
            print(f"SRTP Receiver em modo {args.mode.upper()} - Escutando na porta {args.port}...")
            sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
            sock.bind(('0.0.0.0', args.port))
            
            # Aguarda handshake
            syn_packet, client_addr = sock.recvfrom(HEADER_SIZE)
            valid, _ = verify_crc32(syn_packet)
            
            if valid:
                header = parse_header(syn_packet)
                if protocol_mode == ProtocolMode.SAW:
                    receiver = StopAndWaitReceiver()
                elif protocol_mode == ProtocolMode.GBN:
                    receiver = GoBackNReceiver(args.window)
                else:
                    receiver = SelectiveRepeatReceiver(args.window)
                
                # Processa SYN
                window = receiver.process_handshake(syn_packet, is_initiator=False)
                if window:
                    # Envia SYN+ACK
                    syn_ack_packet = receiver.create_syn_ack_packet(header.length, window)
                    sock.sendto(syn_ack_packet, client_addr)
                    
                    # Aguarda ACK
                    ack_packet, _ = sock.recvfrom(HEADER_SIZE)
                    valid_ack, _ = verify_crc32(ack_packet)
                    
                    if valid_ack:
                        receiver.connected = True
                        print(f"Handshake concluído. Janela negociada: {window} pacotes")
                        
                        # Recebe arquivo
                        output_file = f"received_file_{int(time.time())}"
                        success = receiver.receive_file(sock, output_file)
                        
                        if success:
                            file_hash = compute_file_hash(output_file)
                            print(f"Hash SHA-256 do arquivo: {file_hash}")
                            
                            # Recebe FIN
                            receive_fin(sock, client_addr)
            
            sock.close()
        
        else:
            # Modo sender
            print(f"SRTP Sender em modo {args.mode.upper()} - Conectando a {args.host}:{args.port}...")
            
            if protocol_mode == ProtocolMode.SAW:
                sender = StopAndWaitSender()
            
            sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
            
            # Envía SYN
            syn_packet = sender.create_syn_packet(args.window)
            sock.sendto(syn_packet, (args.host, args.port))
            
            # Aguarda SYN+ACK
            sock.settimeout(TIMEOUT)
            syn_ack_packet, server_addr = sock.recvfrom(HEADER_SIZE)
            
            valid, _ = verify_crc32(syn_ack_packet)
            if valid:
                window = sender.process_handshake(syn_ack_packet, is_initiator=True)
                if window:
                    # Envia ACK
                    ack_packet = sender.create_ack_packet()
                    sock.sendto(ack_packet, server_addr)
                    
                    # Chama o bind para escutar ACKs na porta P+1
                    sock.bind(('0.0.0.0', args.port + 1))
                    
                    print(f"Handshake concluído. Janela negociada: {window} pacotes")
                    
                    # Envia arquivo
                    success = sender.send_file(sock, server_addr, args.file)
                    
                    if success:
                        file_hash = compute_file_hash(args.file)
                        print(f"Hash SHA-256 do arquivo: {file_hash}")
                        
                        # Encerra conexão
                        send_fin(sock, server_addr)
            
            sock.close()
    
    except Exception as e:
        print(f"Erro: {e}")
        import traceback
        traceback.print_exc()

# Exemplo de uso
print("\\nExemplo de uso:")
print("  Receiver: python srtp.py --listen --port 6000 --mode saw")
print("  Sender:   python srtp.py --host 127.0.0.1 --port 6000 --file arquivo.bin --mode saw")
print("\\nImplementação SRTP Completa!")
print("="*60)

In [ ]:
# Teste básico do protocolo SRTP (modo simulado, sem sockets reais)

def test_header_construction():
    """Testa construção e parse do cabeçalho SRTP."""
    print("\\n=== TESTE 1: Construção de Cabeçalho ===")
    
    # Cria cabeçalho com SYN
    syn_header = build_header(seq=0, ack=0, length=16, syn=True, fin=False, 
                             ack_flag=False, nack=False)
    print(f"SYN header (sem CRC): {syn_header.hex()}")
    
    # Cria cabeçalho com SYN+ACK
    syn_ack_header = build_header(seq=0, ack=0, length=8, syn=True, fin=True,
                                 ack_flag=True, nack=False)
    syn_ack_packet = add_crc_to_packet(syn_ack_header)
    print(f"SYN+ACK packet (com CRC): {syn_ack_packet.hex()}")
    
    # Parse
    parsed = parse_header(syn_ack_packet)
    print(f"Parsed SYN+ACK: syn={parsed.syn}, fin={parsed.fin}, ack_flag={parsed.ack_flag}, "
          f"length={parsed.length}")

def test_crc_detection():
    """Testa detecção de corrupção por CRC32."""
    print("\\n=== TESTE 2: Detecção de Corrupção por CRC32 ===")
    
    # Pacote válido
    header = build_header(seq=1, ack=0, length=10, syn=False, fin=False,
                         ack_flag=False, nack=False)
    payload = b'Hello Test'
    valid_packet = add_crc_to_packet(header, payload)
    
    valid, extracted = verify_crc32(valid_packet)
    print(f"Pacote válido: CRC OK = {valid}, Payload = {extracted}")
    
    # Pacote corrompido (altera um byte)
    corrupted_packet = bytearray(valid_packet)
    corrupted_packet[6] ^= 0xFF  # Flipa bits no meio do CRC
    corrupted_packet = bytes(corrupted_packet)
    
    valid_corrupted, _ = verify_crc32(corrupted_packet)
    print(f"Pacote corrompido: CRC OK = {valid_corrupted}")

def test_connection_flow():
    """Testa fluxo de conexão (sem I/O real)."""
    print("\\n=== TESTE 3: Fluxo de Handshake ===")
    
    initiator = StopAndWaitSender()
    receiver = StopAndWaitReceiver()
    
    # Initiator envia SYN
    syn_pkt = initiator.create_syn_packet(window_size=4)
    print(f"Initiator envia SYN (tamanho: {len(syn_pkt)} bytes)")
    
    # Receiver processa SYN
    window = receiver.process_handshake(syn_pkt, is_initiator=False)
    print(f"Receiver processa SYN: janela negociada = {window}")
    
    # Receiver envia SYN+ACK
    syn_ack_pkt = receiver.create_syn_ack_packet(remote_window=4, receiver_window=8)
    print(f"Receiver envia SYN+ACK (tamanho: {len(syn_ack_pkt)} bytes)")
    
    # Initiator processa SYN+ACK
    window = initiator.process_handshake(syn_ack_pkt, is_initiator=True)
    print(f"Initiator processa SYN+ACK: janela negociada = {window}")
    
    # Initiator envia ACK
    ack_pkt = initiator.create_ack_packet()
    print(f"Initiator envia ACK de confirmação (tamanho: {len(ack_pkt)} bytes)")
    print(f"Handshake concluído: {initiator.connected or receiver.connected}")

# Executa testes
test_header_construction()
test_crc_detection()
test_connection_flow()

print("\\n" + "="*60)
print("SRTP T2 - Notebook de Implementação Completo!")
print("Próximas etapas:")
print("1. Implementar testes com Socket real (loopback)")
print("2. Adicionar métricas de throughput e retransmissões")
print("3. Integrar Wireshark para captura de pacotes")
print("4. Testar cenários de latência, perda e reordenação")
print("="*60)